# Educational Plattform Panda Robot
Author: uninh, uqwfa, ultck

## Controller_Task
### Position_Controller

In this task, you will learn... 
>
> - how to work with a position controller to move the joints of a robot to specific positions.
> - how to start the position controller
> - how to create a message for the position controller
> - how to send the target position and check the movement

In [ ]:
##### DO NOT CHANGE #####

import rospy
from controller_manager_msgs.srv import LoadController, SwitchController, SwitchControllerRequest, UnloadController
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
from moveit_commander import roscpp_shutdown

##### DO NOT CHANGE #####

The following code cell loads and activates a new controller in ROS after a potentially conflicting controller has been unloaded.

First, the new controller and the conflicting controller are defined.

Now the conflicting controller will load. First, the /controller_manager/switch_controller service is used to stop the controller. Then the /controller_manager service/unload_controller is used to completely unload. If any of the service calls fail, an error message is issued.

It then waits for the /controller_manager/load_controller service to load the new controller. The service is called and it checks that the controller has been loaded successfully.

Finally, wait for the /controller_manager/switch_controller service to activate the new controller. The service is called and it checks that the controller has been successfully started.

In [ ]:
##### DO NOT CHANGE #####

controller_name = 'position_joint_trajectory_controller'
conflicting_controller = 'effort_joint_trajectory_controller'

# Service for unloading the conflicting controller
rospy.wait_for_service('/controller_manager/switch_controller')
try:
    switch_controller = rospy.ServiceProxy('/controller_manager/switch_controller', SwitchController)
    req = SwitchControllerRequest()
    req.stop_controllers = [conflicting_controller]
    req.strictness = SwitchControllerRequest.BEST_EFFORT
    response = switch_controller(req)
    assert response.ok, "Controller konnte nicht entladen werden."
except rospy.ServiceException as e:
    rospy.logerr("Service call failed: %s" % e)

rospy.wait_for_service('/controller_manager/unload_controller')
try:
    unload_srv = rospy.ServiceProxy('/controller_manager/unload_controller', UnloadController)
    unload_resp = unload_srv(conflicting_controller)
    assert unload_resp.ok, "Controller konnte nicht entladen werden."
except rospy.ServiceException as e:
    rospy.logerr("Service call failed: %s" % e)

# Service for loading the controller
rospy.wait_for_service('/controller_manager/load_controller')
load_service = rospy.ServiceProxy('/controller_manager/load_controller', LoadController)
response = load_service(controller_name)
assert response.ok, "Controller konnte nicht geladen werden."

# Sevice for switching the controller
rospy.wait_for_service('/controller_manager/switch_controller')
switch_service = rospy.ServiceProxy('/controller_manager/switch_controller', SwitchController)
switch_req = SwitchControllerRequest()
switch_req.start_controllers = [controller_name]
switch_req.strictness = SwitchControllerRequest.BEST_EFFORT
response = switch_service(switch_req)
assert response.ok, "Controller konnte nicht gestartet werden."

##### DO NOT CHANGE #####

Now the ROS node is initialized, a publisher is created for the command topic and waited until the publisher is created.

The publisher sends messages of the type JointTrajectory to the topic /position_joint_trajectory_controller/command. The parameter queue_size=10 sets the size of the message queue. This means that up to 10 messages are held in the queue before old messages are discarded.

In [ ]:
##### DO NOT CHANGE #####

# Initialisiere ROS
rospy.init_node('position_control', anonymous=True)

# Publisher für das Command-Topic
pub = rospy.Publisher('/position_joint_trajectory_controller/command', JointTrajectory, queue_size=10)

# Warte, bis der Publisher bereit ist
rospy.sleep(1)

##### DO NOT CHANGE #####

The position controller receives messages of the type trajectory_msgs/JointTrajectory.
This message contains joint_names, a list of the joints to be controlled, and points, a list of positions to control the joints.

In [ ]:
##### DO NOT CHANGE #####

# Erstelle die Nachricht
traj_msg = JointTrajectory()
traj_msg.joint_names = ['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7']

# Definiere die Zielpositionen
point = JointTrajectoryPoint()
# Hier wird die Zielpositionen (in Radiant) für jedes Gelenk als eine Liste hinzugefügt
point.positions = [0.5, -0.5, 0.0, 0.0, 0.5, -0.5, 0.0]
# Nun wird die Zeit festgelegt, bis die Bewegung abgeschlossen sein soll
point.time_from_start = rospy.Duration(2.0)  

# Zuletzt wird der Punkt zur Trajektorie hinzugefügt
traj_msg.points.append(point)

# und die Nachricht wird gesendet
pub.publish(traj_msg)

##### DO NOT CHANGE #####

### Task:

Now you have to define two more target positions yourself. We reach the first target position before the robot moves to the second target position.

#### What you need to do:


##### Create the message:

Start by creating a JointTrajectory message.
Next, define the joints you want to move by entering the names in the joint_names list of the JointTrajectory message.

##### Add target positions:

Create a JointTrajectoryPoint to define the target positions for each joint and complete the following fields:

- positions: The target positions in Radiant (e.g. [0.5, -0.5, 0.0])

- time_from_start: The time it takes for the robot to reach the positions

Now add the point to the trajectory and do the same for the second point as for the first.


##### Send the message:

Post the message to the topic /position_joint_trajectory_controller/command.

In [ ]:
##### YOUR CODE HERE #####

# Nachricht erstellen
traj_msg = JointTrajectory()
traj_msg.joint_names = ['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7']

# Erste Zielposition definieren
point1 = JointTrajectoryPoint()
point1.positions = [0.6, -0.6, 0.2, 0.2, 0.5, -0.5, 0.2]
point1.time_from_start = rospy.Duration(2.0)
traj_msg.points.append(point1)

# Zweite Zielposition definieren
point2 = JointTrajectoryPoint()
point2.positions = [0.7, -0.2, -0.5, -0.2, -0.5, 0.7, -0.2]
point2.time_from_start = rospy.Duration(5.0)
traj_msg.points.append(point2)

# Nachricht senden
pub.publish(traj_msg)

##### YOUR CODE HERE #####

### Cleaning up and quitting the ROS node
At the end, the ROS node is shut down and the resources are released.

In [ ]:
##### DO NOT CHANGE #####

# Shutdown the ROS node
rospy.signal_shutdown("Task completed")
roscpp_shutdown()

##### DO NOT CHANGE #####